# Auditoria final do dataset — Sprint 5 (P3)

Esta auditoria confere **os dados que o modelo oficial realmente usou**. Ela responde a três perguntas:

1. **O modelo "colou" na prova?** Algum paciente do teste também apareceu no treino?
2. **Quais dados foram usados?** Quantos pacientes, nódulos e recortes, e quantos são malignos.
3. **O que ficou de fora, e por quê?**

Além disso, cobre os outros pontos da Sprint 5: a origem dos 48 pacientes, as imagens da explicabilidade, a regra de "maligno" (D6) e se a ficha do radiologista entrega a resposta (D5).

> **A auditoria só lê os arquivos.** Ela não muda nada no treino, nas previsões nem no teste.

## Como rodar (André)

1. Crie uma pasta `dados_andre/` ao lado deste notebook e coloque nela:
   - `divisao_pacientes.csv`: em qual fold cada paciente é teste;
   - o CSV das **previsões oficiais**: uma linha por recorte, com paciente, fold, rótulo e média de malignidade (se tiver nódulo, fatia e diâmetro, melhor);
   - as listas de casos da explicabilidade: os 160 casos de XAI, os 24 do Adebayo e os 16 exemplos.
2. Na célula **CONFIGURAÇÃO** logo abaixo, confira os nomes dos arquivos e aponte `PASTA_LIDC` para a pasta do LIDC-IDRI (DICOM + XML), se ela estiver na sua máquina.
3. Use **Run All**. Tudo que for gerado vai para a pasta `evidencias/`. Mande essa pasta de volta para o Matheus.

Se faltar algum arquivo, a seção correspondente mostra **PENDENTE** e o notebook continua rodando até o fim. Se uma coluna não for encontrada, o aviso diz qual nome ajustar em `COLUNAS`.

**Precisa só de:** Python 3.9+, pandas e numpy. O `pydicom` é opcional: sem ele, o tamanho dos nódulos sai em pixels em vez de mm.

## Configuração

É a única célula que pode precisar de ajuste.

In [1]:
# ========================= CONFIGURAÇÃO =========================
from pathlib import Path

PASTA_BASE  = Path.cwd()                    # pasta onde está este notebook
PASTA_DADOS = PASTA_BASE / "dados_andre"    # arquivos do treino oficial
PASTA_SAIDA = PASTA_BASE / "evidencias"     # tudo que a auditoria gera vai para cá

# Se o nome não bater, o notebook procura em PASTA_DADOS um CSV com uma das palavras da lista.
ARQ_DIVISAO   = PASTA_DADOS / "divisao_pacientes.csv"
ARQ_PREVISOES = PASTA_DADOS / "previsoes_oficiais.csv"
ARQS_XAI = {                                # listas de casos da explicabilidade
    "xai_160":     PASTA_DADOS / "casos_xai.csv",
    "adebayo_24":  PASTA_DADOS / "casos_adebayo.csv",
    "exemplos_16": PASTA_DADOS / "exemplos.csv",
}

# Opcional: pasta com o LIDC-IDRI original (DICOM + XML). None = pula as partes que dependem dela.
PASTA_LIDC = Path(r"D:\datasetLIDC_completo\LIDC-IDRI")   # dataset completo (~130 GB) da máquina do Gabriel

# [Adaptação Gabriel] Nesta máquina os XMLs estão todos em PASTA_LIDC/_xml_official (pacote oficial do TCIA),
# sem o ID do paciente no caminho e sem DICOM na mesma pasta. O índice abaixo (gerado pelo notebook 03) liga
# série/exame -> paciente e série -> um DICOM, para achar o paciente e o tamanho do pixel.
INDICE_DICOM = PASTA_BASE.parent / "outputs" / "sprint3" / "dicom_index.csv"

# Regras do treino oficial (rascunho do André, Sprint 5)
LIMIAR_MALIGNO  = 3.5                       # média dos radiologistas >= 3,5 -> maligno
AMBIGUO         = (2.5, 3.5)                # média estritamente entre esses valores -> ambíguo (fora da visão principal)
SHA256_ESPERADO = "d6c22d88d1c0"            # impressão digital da divisão anotada pelo André

# Nomes das colunas: deixe vazio para detectar sozinho, ou force, ex.: {"paciente": "PatientID"}
COLUNAS = {}

## Preparação

Funções de apoio. Não precisa mexer aqui.

In [2]:
import hashlib, platform, re, sys, datetime, xml.etree.ElementTree as ET
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

RESUMO = []   # (seção, status, frase) — vira o resumo_auditoria.md no final

def registrar(secao, status, frase):
    """status: OK, ALERTA, PENDENTE ou INFO."""
    RESUMO.append((secao, status, frase))
    print(f"[{status}] {frase}")

def salvar(df, nome):
    caminho = PASTA_SAIDA / nome
    df.to_csv(caminho, index=False, encoding="utf-8")
    print(f"   -> salvo: evidencias/{nome} ({len(df)} linhas)")

def achar_arquivo(caminho, palavras):
    """Usa o caminho configurado; se não existir, procura um CSV em PASTA_DADOS com uma das palavras no nome."""
    caminho = Path(caminho)
    if caminho.exists():
        return caminho
    if PASTA_DADOS.exists():
        achados = [p for p in sorted(PASTA_DADOS.glob("*.csv"))
                   if any(w in p.stem.lower() for w in palavras)]
        if len(achados) == 1:
            print(f"   (usando {achados[0].name} no lugar de {caminho.name})")
            return achados[0]
        if len(achados) > 1:
            print(f"   Mais de um candidato para {caminho.name}: {[p.name for p in achados]} — ajuste na CONFIGURAÇÃO.")
    return None

def ler_csv(caminho):
    """Aceita vírgula ou ponto e vírgula, decidido pela linha do cabeçalho."""
    with open(caminho, encoding="utf-8-sig", errors="replace") as f:
        cabecalho = f.readline()
    sep = ";" if cabecalho.count(";") > cabecalho.count(",") else ","
    return pd.read_csv(caminho, sep=sep, encoding="utf-8-sig")

CANDIDATOS = {
    "paciente": ["patient_id", "paciente", "patient", "pid", "patientid", "id_paciente"],
    "fold":     ["fold", "fold_id", "cv_fold", "test_fold", "fold_teste", "fold_test"],
    "papel":    ["split", "papel", "conjunto", "subset", "role", "particao"],
    "rotulo":   ["label", "rotulo", "y_true", "target", "y", "classe"],
    "media":    ["malignancy_mean", "media_malignidade", "mean_malignancy", "mal_mean", "malignancy", "media"],
    "prob":     ["prob", "y_prob", "prob_oficial", "prob_ensemble", "probability", "score", "pred_prob", "p_maligno"],
    "caso":     ["case_id", "caso_id", "caso", "sample_id", "row_id", "id", "test_row_id"],
    "nodulo":   ["nodule_id", "nodulo_id", "nodule_uid", "nodulo", "nodule", "cluster_id"],
    "exame":    ["study_uid", "study_instance_uid", "studyinstanceuid", "exame"],
    "serie":    ["series_uid", "series_instance_uid", "seriesinstanceuid", "serie"],
    "fatia":    ["sop_uid", "sop_instance_uid", "sopinstanceuid", "image_sop_uid", "fatia", "slice_idx", "slice_index", "slice"],
    "diametro": ["diameter_mm", "diametro_mm", "nodule_diameter_mm", "diameter", "diametro"],
    "grupo":    ["grupo", "group", "categoria", "tipo", "outcome", "resultado"],
}

def achar_coluna(df, papel):
    """Devolve o nome da coluna que faz o papel pedido, ou None."""
    if df is None:
        return None
    forcada = COLUNAS.get(papel)
    if forcada:
        if forcada not in df.columns:
            print(f"   ATENÇÃO: COLUNAS['{papel}'] = '{forcada}', mas essa coluna não existe. Colunas: {list(df.columns)}")
            return None
        return forcada
    minusculas = {c.lower().strip(): c for c in df.columns}
    for cand in CANDIDATOS[papel]:
        if cand in minusculas:
            return minusculas[cand]
    return None

def normalizar_paciente(s):
    """'LIDC-IDRI-0001', '0001', 1 -> 'LIDC-IDRI-0001'."""
    s = str(s).strip()
    m = re.search(r"(\d{1,4})\s*$", s)
    return f"LIDC-IDRI-{int(m.group(1)):04d}" if m else s

def auc(y, score):
    """Área sob a curva ROC (Mann-Whitney, trata empates). 0,5 = chute; 1 = acerta sempre."""
    y = np.asarray(y, dtype=float); score = np.asarray(score, dtype=float)
    ok = ~(np.isnan(y) | np.isnan(score)); y, score = y[ok], score[ok]
    n1, n0 = (y == 1).sum(), (y == 0).sum()
    if n1 == 0 or n0 == 0:
        return np.nan
    postos = pd.Series(score).rank().to_numpy()
    return (postos[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

def sha256_arquivo(caminho):
    h = hashlib.sha256()
    with open(caminho, "rb") as f:
        for bloco in iter(lambda: f.read(1 << 20), b""):
            h.update(bloco)
    return h.hexdigest()

print("Python", platform.python_version(), "| pandas", pd.__version__, "| numpy", np.__version__)

Python 3.13.2 | pandas 3.0.5 | numpy 2.5.2


## Carregar os arquivos do treino oficial

Lê a divisão e as previsões e mostra qual coluna foi reconhecida para cada informação. **Confira essa tabela antes de seguir:** se alguma coluna saiu errada, ajuste em `COLUNAS`.

In [3]:
arq_div  = achar_arquivo(ARQ_DIVISAO, ["divis", "split", "fold"])
arq_prev = achar_arquivo(ARQ_PREVISOES, ["previs", "pred", "manifest", "oof"])

divisao  = ler_csv(arq_div)  if arq_div  else None
prev     = ler_csv(arq_prev) if arq_prev else None

for nome, df, arq in [("divisão", divisao, arq_div), ("previsões", prev, arq_prev)]:
    if df is None:
        print(f"PENDENTE: arquivo de {nome} não encontrado em {PASTA_DADOS}")
    else:
        print(f"{nome}: {arq.name} — {len(df)} linhas, colunas: {list(df.columns)}")

mapa = pd.DataFrame([
    {"informação": papel,
     "coluna na divisão": achar_coluna(divisao, papel) or "-",
     "coluna nas previsões": achar_coluna(prev, papel) or "-"}
    for papel in CANDIDATOS
])
mapa

PENDENTE: arquivo de divisão não encontrado em C:\Users\Administrator\Documents\Biopark\Projeto-Integrador-3\PI3-Grupo-03\auditoria_dataset\dados_andre
PENDENTE: arquivo de previsões não encontrado em C:\Users\Administrator\Documents\Biopark\Projeto-Integrador-3\PI3-Grupo-03\auditoria_dataset\dados_andre


,informação,coluna na divisão,coluna nas previsões
0,paciente,-,-
1,fold,-,-
2,papel,-,-
3,rotulo,-,-
4,media,-,-
5,prob,-,-
6,caso,-,-
7,nodulo,-,-
8,exame,-,-
9,serie,-,-


In [4]:
# Monta duas tabelas padronizadas, usadas no resto do notebook:
#   PAPEIS:  paciente | fold | papel  (papel = teste, validacao, treino ou treino_ou_validacao)
#   LINHAS:  uma linha por recorte, com nomes de coluna padronizados
PAPEIS, LINHAS, TEM_VALIDACAO = None, None, False

if divisao is not None:
    c_pac = achar_coluna(divisao, "paciente")
    c_fold = achar_coluna(divisao, "fold")
    c_papel = achar_coluna(divisao, "papel")
    colunas_fold_largas = [c for c in divisao.columns if "fold" in c.lower() and c != c_fold]
    if c_pac is None:
        print("PENDENTE: não achei a coluna de paciente na divisão — ajuste COLUNAS['paciente'].")
    elif c_fold is not None and c_papel is None:
        # formato "longo simples": uma linha por paciente com o fold em que ele é teste
        PAPEIS = pd.DataFrame({"paciente": divisao[c_pac].map(normalizar_paciente),
                               "fold": divisao[c_fold].astype(str), "papel": "teste"})
    elif c_fold is not None and c_papel is not None:
        # formato longo completo: paciente | fold | papel
        papel_txt = divisao[c_papel].astype(str).str.lower()
        PAPEIS = pd.DataFrame({"paciente": divisao[c_pac].map(normalizar_paciente),
                               "fold": divisao[c_fold].astype(str),
                               "papel": np.select([papel_txt.str.contains("test"), papel_txt.str.contains("val")],
                                                  ["teste", "validacao"], "treino")})
    elif colunas_fold_largas:
        # formato largo: uma coluna por fold com 'train'/'val'/'test'
        longo = divisao.melt(id_vars=[c_pac], value_vars=colunas_fold_largas, var_name="fold", value_name="papel")
        papel_txt = longo["papel"].astype(str).str.lower()
        PAPEIS = pd.DataFrame({"paciente": longo[c_pac].map(normalizar_paciente), "fold": longo["fold"].astype(str),
                               "papel": np.select([papel_txt.str.contains("test"), papel_txt.str.contains("val")],
                                                  ["teste", "validacao"], "treino")})
    else:
        print("PENDENTE: não reconheci o formato da divisão (sem coluna de fold). Ajuste COLUNAS['fold'].")

if PAPEIS is not None:
    TEM_VALIDACAO = (PAPEIS["papel"] == "validacao").any()
    print(f"Divisão: {PAPEIS['paciente'].nunique()} pacientes, folds = {sorted(PAPEIS['fold'].unique())}")
    print("Validação separada no arquivo:", "sim" if TEM_VALIDACAO else "não (só dá para conferir teste x resto)")

if prev is not None:
    LINHAS = pd.DataFrame(index=prev.index)
    for papel in CANDIDATOS:
        c = achar_coluna(prev, papel)
        if c is not None:
            LINHAS[papel] = prev[c]
    if "paciente" in LINHAS:
        LINHAS["paciente"] = LINHAS["paciente"].map(normalizar_paciente)
    if "fold" in LINHAS:
        LINHAS["fold"] = LINHAS["fold"].astype(str)
    elif PAPEIS is not None and "paciente" in LINHAS:
        # sem fold nas previsões: usa o fold em que o paciente é teste, segundo a divisão
        fold_teste = PAPEIS[PAPEIS["papel"] == "teste"].drop_duplicates("paciente").set_index("paciente")["fold"]
        LINHAS["fold"] = LINHAS["paciente"].map(fold_teste)
    if "media" in LINHAS:
        LINHAS["media"] = pd.to_numeric(LINHAS["media"], errors="coerce")
        LINHAS["ambiguo"] = (LINHAS["media"] > AMBIGUO[0]) & (LINHAS["media"] < AMBIGUO[1])
    if "rotulo" in LINHAS:
        LINHAS["rotulo"] = pd.to_numeric(LINHAS["rotulo"], errors="coerce")
    if "nodulo" in LINHAS and "paciente" in LINHAS:
        LINHAS["chave_nodulo"] = LINHAS["paciente"] + "|" + LINHAS["nodulo"].astype(str)
    print(f"Previsões: {len(LINHAS)} recortes, colunas reconhecidas: {[c for c in LINHAS.columns]}")

## 1. O modelo "colou" na prova? (separação por paciente)

**O que é verificado.** O André usou **validação cruzada com 5 folds**. É um rodízio: os pacientes são separados em 5 grupos e o modelo é treinado 5 vezes. Em cada vez, um grupo é o teste e os outros 4 são usados para treinar. Para o resultado valer:

- cada paciente precisa ser **teste em exatamente um fold**;
- em cada fold, **nenhum paciente do teste** pode estar no treino.

Também conferimos exame, série, fatia e nódulo: o mesmo exame não pode aparecer em dois folds, nem com dois pacientes diferentes.

**Impressão digital do arquivo.** O sha256 é um código calculado a partir do conteúdo. Se ele bater com o que o André anotou (`d6c22d88d1c0`), este é **o mesmo arquivo** que o treino usou.

In [5]:
SEC = "1. Separação"
if PAPEIS is None:
    registrar(SEC, "PENDENTE", "Sem o arquivo da divisão, não dá para conferir a separação.")
else:
    sha = sha256_arquivo(arq_div)
    if sha.startswith(SHA256_ESPERADO.lower()):
        registrar(SEC, "OK", f"A impressão digital da divisão bate com a anotada pelo André ({sha[:12]}).")
    else:
        registrar(SEC, "ALERTA", f"Impressão digital diferente: {sha[:12]} (esperado {SHA256_ESPERADO}). "
                                 "Este pode não ser o arquivo usado no treino.")

    teste = PAPEIS[PAPEIS["papel"] == "teste"]
    vezes_teste = teste.groupby("paciente")["fold"].nunique()
    todos = PAPEIS["paciente"].unique()
    nunca_teste = sorted(set(todos) - set(vezes_teste.index))
    mais_de_uma = vezes_teste[vezes_teste > 1]
    por_fold = teste.groupby("fold")["paciente"].nunique().rename("pacientes no teste").reset_index()
    display(por_fold)

    if len(mais_de_uma) == 0 and not nunca_teste:
        registrar(SEC, "OK", f"Cada um dos {len(todos)} pacientes é teste em exatamente 1 fold.")
    else:
        registrar(SEC, "ALERTA", f"{len(mais_de_uma)} pacientes são teste em mais de um fold e "
                                 f"{len(nunca_teste)} nunca são teste.")

    linhas_inter = []
    for fold in sorted(PAPEIS["fold"].unique()):
        f = PAPEIS[PAPEIS["fold"] == fold]
        te = set(teste[teste["fold"] == fold]["paciente"])
        if TEM_VALIDACAO:
            va = set(f[f["papel"] == "validacao"]["paciente"])
            tr = set(f[f["papel"] == "treino"]["paciente"])
        else:   # só o teste é conhecido: o "resto" são os pacientes dos outros folds
            va, tr = set(), set(todos) - te
        linhas_inter.append({"fold": fold, "teste": len(te), "validação": len(va) if TEM_VALIDACAO else "não salvo",
                             "treino": len(tr), "teste ∩ treino": len(te & tr),
                             "teste ∩ validação": len(te & va) if TEM_VALIDACAO else "-",
                             "validação ∩ treino": len(va & tr) if TEM_VALIDACAO else "-"})
    inter = pd.DataFrame(linhas_inter)
    display(inter)
    salvar(inter, "1_separacao_por_fold.csv")
    soma = sum(int(x) for c in ["teste ∩ treino", "teste ∩ validação", "validação ∩ treino"]
               for x in inter[c] if x != "-")
    if soma == 0:
        registrar(SEC, "OK", "Nenhum paciente aparece ao mesmo tempo no teste e no treino de um fold.")
    else:
        registrar(SEC, "ALERTA", f"{soma} sobreposições de paciente entre grupos — ver 1_separacao_por_fold.csv.")
    if not TEM_VALIDACAO:
        registrar(SEC, "PENDENTE", "A divisão não traz quem foi validação dentro de cada fold; "
                                   "a separação validação x treino não pôde ser conferida.")

if LINHAS is not None and "fold" in LINHAS:
    niveis = [n for n in ["exame", "serie", "fatia", "nodulo"] if n in LINHAS]
    linhas_niveis = []
    for nivel in niveis:
        ids = LINHAS[[nivel, "fold", "paciente"]].dropna().astype(str)
        globais = ids.groupby(nivel)["paciente"].nunique().max() == 1 or nivel != "nodulo"
        if nivel == "nodulo" and not globais:
            # o ID de nódulo só vale dentro do paciente: a checagem por paciente já cobre
            linhas_niveis.append({"nível": nivel, "IDs": ids[nivel].nunique(),
                                  "em mais de 1 fold": "coberto pela checagem de paciente", "em mais de 1 paciente": "-"})
            continue
        linhas_niveis.append({"nível": nivel, "IDs": ids[nivel].nunique(),
                              "em mais de 1 fold": int((ids.groupby(nivel)["fold"].nunique() > 1).sum()),
                              "em mais de 1 paciente": int((ids.groupby(nivel)["paciente"].nunique() > 1).sum())})
    if linhas_niveis:
        niv = pd.DataFrame(linhas_niveis)
        display(niv)
        salvar(niv, "1_separacao_por_nivel.csv")
        problemas = sum(int(v) for c in ["em mais de 1 fold", "em mais de 1 paciente"]
                        for v in niv[c] if isinstance(v, (int, np.integer)))
        registrar(SEC, "OK" if problemas == 0 else "ALERTA",
                  f"Checagem além do paciente ({', '.join(niveis)}): {problemas} IDs repetidos entre folds ou pacientes.")
    else:
        registrar(SEC, "INFO", "As previsões não trazem exame, série, fatia nem nódulo; a checagem ficou só por paciente.")

[PENDENTE] Sem o arquivo da divisão, não dá para conferir a separação.


**Como ler.** O esperado é **zero** em todas as colunas com "∩" e em "mais de 1 fold". Zero significa que o modelo nunca viu na prova um paciente que estudou no treino.

## 2. Quais dados foram usados? (composição)

**O que é verificado.** Contamos pacientes, nódulos e recortes em cada fold e quanto disso é maligno. O mesmo dado aparece de dois jeitos:

- **por recorte:** cada fatia de cada nódulo conta 1 (é o que o modelo vê);
- **por nódulo:** cada nódulo conta 1, não importa em quantas fatias ele aparece.

Esses dois números costumam ser **bem diferentes**: os nódulos malignos são maiores, aparecem em mais fatias e por isso "pesam" mais na contagem por recorte.

**Duas visões**, como no rascunho do André:

- **todos:** todos os recortes;
- **principal:** sem os nódulos **ambíguos** (média das notas entre 2,5 e 3,5).

No fim, os números são comparados com os que o André publicou no rascunho dele.

In [6]:
SEC = "2. Composição"
if LINHAS is None or "paciente" not in LINHAS:
    registrar(SEC, "PENDENTE", "Sem o arquivo de previsões (com paciente), não dá para montar a composição.")
else:
    if "rotulo" not in LINHAS and "media" in LINHAS:
        LINHAS["rotulo"] = (LINHAS["media"] >= LIMIAR_MALIGNO).astype(int)
        print("Sem coluna de rótulo: rótulo calculado pela regra média >= 3,5.")

    def compor(df, visao):
        linhas = []
        grupos = list(df.groupby("fold")) if "fold" in df else []
        for fold, g in grupos + [("TOTAL", df)]:
            l = {"visão": visao, "fold": fold, "pacientes": g["paciente"].nunique(), "recortes": len(g)}
            if "exame" in g: l["exames"] = g["exame"].nunique()
            if "chave_nodulo" in g: l["nódulos"] = g["chave_nodulo"].nunique()
            if "rotulo" in g:
                l["% maligno (recortes)"] = round(100 * g["rotulo"].mean(), 1)
                if "chave_nodulo" in g:
                    l["% maligno (nódulos)"] = round(100 * g.groupby("chave_nodulo")["rotulo"].max().mean(), 1)
            linhas.append(l)
        return linhas

    comp = compor(LINHAS, "todos")
    if "ambiguo" in LINHAS:
        comp += compor(LINHAS[~LINHAS["ambiguo"]], "principal")
    comp = pd.DataFrame(comp)
    display(comp)
    salvar(comp, "2_composicao_por_fold.csv")
    registrar(SEC, "INFO", f"Composição salva: {LINHAS['paciente'].nunique()} pacientes, {len(LINHAS)} recortes (visão todos).")

    # Conferência com os números do rascunho do André
    esperados = {"pacientes": 757, "recortes (todos)": 13446, "nódulos (todos)": 2306, "nódulos malignos (todos)": 449,
                 "nódulos ambíguos": 910, "recortes (principal)": 8823, "nódulos (principal)": 1396,
                 "recortes malignos (principal)": 4586}
    obtidos = {"pacientes": LINHAS["paciente"].nunique(), "recortes (todos)": len(LINHAS)}
    if "ambiguo" in LINHAS:
        principal = LINHAS[~LINHAS["ambiguo"]]
        obtidos["recortes (principal)"] = len(principal)
        if "rotulo" in LINHAS:
            obtidos["recortes malignos (principal)"] = int(principal["rotulo"].sum())
    if "chave_nodulo" in LINHAS and "rotulo" in LINHAS:
        nod = LINHAS.groupby("chave_nodulo").agg(rotulo=("rotulo", "max"), ambiguo=("ambiguo", "max")) \
            if "ambiguo" in LINHAS else LINHAS.groupby("chave_nodulo").agg(rotulo=("rotulo", "max"))
        obtidos["nódulos (todos)"] = len(nod)
        obtidos["nódulos malignos (todos)"] = int(nod["rotulo"].sum())
        if "ambiguo" in nod:
            obtidos["nódulos ambíguos"] = int(nod["ambiguo"].sum())
            obtidos["nódulos (principal)"] = int((~nod["ambiguo"].astype(bool)).sum())
    conf = pd.DataFrame([{"número": k, "rascunho do André": v, "obtido aqui": obtidos.get(k, "sem coluna"),
                          "confere?": ("sim" if obtidos.get(k) == v else "NÃO") if k in obtidos else "-"}
                         for k, v in esperados.items()])
    display(conf)
    salvar(conf, "2_conferencia_com_rascunho.csv")
    diverg = conf[conf["confere?"] == "NÃO"]
    if len(diverg):
        registrar(SEC, "ALERTA", f"{len(diverg)} números diferentes do rascunho do André: {', '.join(diverg['número'])}.")
    else:
        registrar(SEC, "OK", "Todos os números que deu para calcular conferem com o rascunho do André.")

[PENDENTE] Sem o arquivo de previsões (com paciente), não dá para montar a composição.


**Como ler.** "% maligno (recortes)" é o que o modelo enxerga; "% maligno (nódulos)" é o que acontece de fato nos pacientes. A tabela de conferência mostra se os números deste arquivo são os mesmos publicados. Um **NÃO** indica arquivo diferente ou regra diferente, e precisa ser explicado antes de usar os números.

## 3. O que ficou de fora, e por quê? (descartes e correspondência)

**O que é verificado.**

- **Funil:** quantos pacientes existem no LIDC, quantos têm nódulo com nota de malignidade e quantos entraram na divisão. Quem **tem nódulo com nota mas não entrou** sumiu sem motivo registrado. Esses casos precisam de explicação.
- **Correspondência:** cada recorte precisa ter um paciente que existe na divisão, um ID único e um rótulo que obedece à regra (média ≥ 3,5 = maligno). Se as previsões trazem a fatia (UID), conferimos também que existe uma anotação de radiologista naquela fatia.
- **Conferência manual:** 3 recortes sorteados são mostrados ao lado das anotações originais, para conferir a olho.

In [7]:
SEC = "3. Descartes"
CARACTERISTICAS = ["subtlety", "internalStructure", "calcification", "sphericity",
                   "margin", "lobulation", "spiculation", "texture", "malignancy"]

def _tag(e):
    return e.tag.split("}")[-1]

def _filho(e, nome):
    for c in e:
        if _tag(c) == nome:
            return c
    return None

def ler_xml_lidc(caminho):
    """Uma linha por anotação (leitor x nódulo) de um XML de TC do LIDC."""
    try:
        raiz = ET.parse(caminho).getroot()
    except ET.ParseError:
        return []
    if "LidcReadMessage" not in _tag(raiz):
        return []                     # XML de raio-X ou outro formato
    m = re.search(r"LIDC-IDRI-\d{4}", str(caminho))
    series = next((e.text for e in raiz.iter() if _tag(e).lower() == "seriesinstanceuid"), None)
    estudo = next((e.text for e in raiz.iter() if _tag(e).lower() == "studyinstanceuid"), None)
    # [Adaptação Gabriel] sem paciente no caminho, usa a série (ou o exame) do XML no índice DICOM
    paciente = m.group(0) if m else SERIE_PACIENTE.get(series) or ESTUDO_PACIENTE.get(estudo)
    linhas, leitor = [], 0
    for sessao in raiz.iter():
        if _tag(sessao) != "readingSession":
            continue
        leitor += 1
        for nod in sessao:
            if _tag(nod) != "unblindedReadNodule":
                continue
            car = _filho(nod, "characteristics")
            l = {"paciente": paciente, "xml": str(caminho), "serie": series, "leitor": leitor,
                 "nodulo_xml": getattr(_filho(nod, "noduleID"), "text", None)}
            for c in CARACTERISTICAS:
                v = getattr(_filho(car, c), "text", None) if car is not None else None
                l[c] = float(v) if v not in (None, "") else np.nan
            sops, maior_area = set(), 0.0
            for roi in nod:
                if _tag(roi) != "roi":
                    continue
                sop = getattr(_filho(roi, "imageSOP_UID"), "text", None)
                if sop: sops.add(sop)
                incl = getattr(_filho(roi, "inclusion"), "text", "TRUE")
                pts = [(float(_filho(ed, "xCoord").text), float(_filho(ed, "yCoord").text))
                       for ed in roi if _tag(ed) == "edgeMap"]
                if len(pts) >= 3 and str(incl).upper() == "TRUE":
                    x, y = np.array(pts).T
                    maior_area = max(maior_area, 0.5 * abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1))))
            l["fatias_sop"] = sops
            l["n_fatias"] = len(sops)
            l["maior_area_px"] = maior_area
            linhas.append(l)
    return linhas

# [Adaptação Gabriel] série/exame -> paciente e série -> um DICOM, a partir do índice do notebook 03
SERIE_PACIENTE, ESTUDO_PACIENTE, SERIE_DCM = {}, {}, {}
if Path(INDICE_DICOM).exists():
    _idx = pd.read_csv(INDICE_DICOM, dtype=str, usecols=["patient_id", "study_uid", "series_uid", "path"])
    SERIE_PACIENTE = dict(zip(_idx["series_uid"], _idx["patient_id"]))
    ESTUDO_PACIENTE = dict(zip(_idx["study_uid"], _idx["patient_id"]))
    SERIE_DCM = _idx.drop_duplicates("series_uid").set_index("series_uid")["path"].to_dict()
    print(f"Índice DICOM: {len(SERIE_PACIENTE):,} séries, {len(ESTUDO_PACIENTE):,} exames, "
          f"{_idx['patient_id'].nunique():,} pacientes")
else:
    print(f"Índice DICOM não encontrado em {INDICE_DICOM}: XMLs sem paciente no caminho ficarão sem paciente.")

ANOT = None
if PASTA_LIDC is None or not Path(PASTA_LIDC).exists():
    print("PASTA_LIDC não configurada: o funil a partir do LIDC original fica PENDENTE.")
else:
    xmls = sorted(Path(PASTA_LIDC).rglob("*.xml"))
    raiz_lidc = Path(PASTA_LIDC)
    pastas_pac = {p.name for p in raiz_lidc.iterdir() if re.fullmatch(r"LIDC-IDRI-\d{4}", p.name)}
    if not pastas_pac:   # apontou para a pasta de cima (ex.: manifest-xxx/): olha um nível abaixo
        pastas_pac = {p.name for p in raiz_lidc.glob("*/*") if re.fullmatch(r"LIDC-IDRI-\d{4}", p.name)}
    ANOT = pd.DataFrame([l for x in xmls for l in ler_xml_lidc(x)])
    print(f"LIDC: {len(pastas_pac)} pastas de paciente, {len(xmls)} XML, {len(ANOT)} anotações lidas.")

if ANOT is not None and len(ANOT):
    com_nota = set(ANOT.loc[ANOT["malignancy"].notna(), "paciente"].dropna())
    com_anot = set(ANOT["paciente"].dropna())
    na_divisao = set(PAPEIS["paciente"]) if PAPEIS is not None else set()
    funil = [("pastas de paciente no LIDC", len(pastas_pac)),
             ("pacientes com alguma anotação", len(com_anot)),
             ("pacientes com nódulo >= 3 mm com nota", len(com_nota))]
    if PAPEIS is not None:
        funil.append(("pacientes na divisão oficial", len(na_divisao)))
    funil = pd.DataFrame(funil, columns=["etapa", "pacientes"])
    display(funil)
    salvar(funil, "3_funil_pacientes.csv")

    if PAPEIS is not None:
        fora = []
        for p in sorted(pastas_pac - na_divisao):
            if p not in com_anot:
                motivo = "sem nenhuma anotação de nódulo"
            elif p not in com_nota:
                motivo = "só nódulos < 3 mm (sem nota de malignidade)"
            else:
                motivo = "SEM MOTIVO REGISTRADO: tem nódulo com nota mas não entrou"
            fora.append({"paciente": p, "motivo": motivo})
        fora = pd.DataFrame(fora, columns=["paciente", "motivo"])
        if len(fora):
            display(fora["motivo"].value_counts().rename("pacientes").to_frame())
        salvar(fora, "3_pacientes_fora_da_divisao.csv")
        sem_motivo = int(fora["motivo"].str.startswith("SEM MOTIVO").sum()) if len(fora) else 0
        registrar(SEC, "OK" if sem_motivo == 0 else "ALERTA",
                  f"{len(fora)} pacientes do LIDC ficaram fora da divisão; {sem_motivo} sem motivo registrado "
                  "(têm nódulo com nota e mesmo assim não entraram).")
        extras = sorted(na_divisao - pastas_pac)
        if extras and len(pastas_pac) > 0:
            registrar(SEC, "INFO", f"{len(extras)} pacientes da divisão não estão na PASTA_LIDC desta máquina "
                                   "(normal se a pasta for parcial).")
    else:
        registrar(SEC, "PENDENTE", "Funil sem a divisão oficial: só mostra o que existe no LIDC.")
else:
    registrar(SEC, "PENDENTE", "Sem a pasta do LIDC original, não dá para montar o funil de pacientes.")

# Correspondência dentro das previsões
if LINHAS is None or "paciente" not in LINHAS:
    registrar(SEC, "PENDENTE", "Sem o arquivo de previsões, a correspondência recorte -> paciente -> rótulo fica pendente.")
else:
    checks = []
    if PAPEIS is not None:
        orfaos = int((~LINHAS["paciente"].isin(set(PAPEIS["paciente"]))).sum())
        checks.append(("recortes de paciente que não está na divisão", orfaos))
        sem_recorte = len(set(PAPEIS["paciente"]) - set(LINHAS["paciente"]))
        checks.append(("pacientes da divisão sem nenhum recorte", sem_recorte))
        if achar_coluna(prev, "fold"):
            fold_teste = PAPEIS[PAPEIS["papel"] == "teste"].drop_duplicates("paciente").set_index("paciente")["fold"]
            checks.append(("recortes previstos por um fold em que o paciente NÃO era teste",
                           int((LINHAS["fold"] != LINHAS["paciente"].map(fold_teste)).sum())))
    if "caso" in LINHAS:
        checks.append(("IDs de recorte repetidos", int(LINHAS["caso"].duplicated().sum())))
    if "rotulo" in LINHAS and "media" in LINHAS:
        regra = (LINHAS["media"] >= LIMIAR_MALIGNO).astype(int)
        checks.append(("rótulos que não obedecem à regra média >= 3,5", int((regra != LINHAS["rotulo"]).sum())))
    if "fatia" in LINHAS and ANOT is not None and len(ANOT) and \
            LINHAS["fatia"].astype(str).str.startswith("1.3.6").mean() > 0.9:
        anotadas = {(p, s) for p, ss in zip(ANOT["paciente"], ANOT["fatias_sop"]) for s in ss}
        presentes = LINHAS[LINHAS["paciente"].isin(set(ANOT["paciente"]))]
        sem_anot = sum((p, str(s)) not in anotadas for p, s in zip(presentes["paciente"], presentes["fatia"]))
        checks.append((f"recortes cuja fatia não tem anotação no XML (de {len(presentes)} conferíveis)", sem_anot))
    checks = pd.DataFrame(checks, columns=["checagem", "quantidade (esperado 0)"])
    display(checks)
    salvar(checks, "3_correspondencia.csv")
    ruins = checks[checks["quantidade (esperado 0)"] > 0]
    registrar(SEC, "OK" if ruins.empty else "ALERTA",
              "Correspondência sem problemas." if ruins.empty else
              "Problemas de correspondência: " + "; ".join(f"{c} = {q}" for c, q in ruins.itertuples(index=False)))

    amostra = LINHAS.sample(min(3, len(LINHAS)), random_state=0)
    print("\nConferência manual: 3 recortes sorteados (seed 0). Confira paciente, nódulo e nota nas anotações originais.")
    display(amostra)
    if ANOT is not None and len(ANOT):
        for p in amostra["paciente"]:
            a = ANOT[ANOT["paciente"] == p][["paciente", "leitor", "nodulo_xml", "malignancy", "n_fatias"]]
            print(f"\nAnotações no XML de {p}:"); display(a.head(12))

Índice DICOM: 650 séries, 650 exames, 642 pacientes


LIDC: 1010 pastas de paciente, 1319 XML, 20383 anotações lidas.


,etapa,pacientes
0,pastas de paciente no LIDC,1010
1,pacientes com alguma anotação,627
2,pacientes com nódulo >= 3 mm com nota,559


   -> salvo: evidencias/3_funil_pacientes.csv (3 linhas)
[PENDENTE] Funil sem a divisão oficial: só mostra o que existe no LIDC.
[PENDENTE] Sem o arquivo de previsões, a correspondência recorte -> paciente -> rótulo fica pendente.


## 4. De onde vêm os 48 pacientes?

**O que é verificado.** Nas Sprints 3 e 4, o grupo usou um **piloto** com 48 pacientes (33 treino, 7 validação, 8 teste). É dele que veio uma das duas acurácias que circulavam. A lista abaixo foi tirada das pastas `Sprint-04/dataset_dividido` e está escrita no notebook para rodar em qualquer máquina.

Vamos ver em quais folds da divisão oficial esses 48 caíram, para mostrar que o piloto é **outro experimento**, e não a base do resultado oficial.

In [8]:
SEC = "4. Origem dos 48"
PILOTO_48 = {   # Sprint-04/dataset_dividido (pacientes LIDC-IDRI-0001 a 0050, menos 0028 e 0032)
    "treino": [1, 4, 5, 10, 11, 12, 13, 14, 17, 19, 20, 21, 22, 23, 24, 25, 26, 27, 30, 31, 33,
               34, 35, 36, 38, 39, 42, 44, 45, 46, 47, 49, 50],
    "validacao": [3, 6, 29, 37, 40, 41, 48],
    "teste": [2, 7, 8, 9, 15, 16, 18, 43],
}
piloto = pd.DataFrame([{"paciente": f"LIDC-IDRI-{n:04d}", "grupo no piloto": g}
                       for g, ns in PILOTO_48.items() for n in ns])
faltam = sorted(set(range(1, 51)) - {n for ns in PILOTO_48.values() for n in ns})
print(f"Piloto: {len(piloto)} pacientes; números de 1 a 50 que não estão nele: {faltam}")

if PAPEIS is None:
    registrar(SEC, "INFO", f"O piloto tem {len(piloto)} pacientes (0001 a 0050, sem {faltam}); "
                           "o cruzamento com os folds oficiais fica PENDENTE.")
else:
    fold_teste = PAPEIS[PAPEIS["papel"] == "teste"].drop_duplicates("paciente").set_index("paciente")["fold"]
    piloto["fold oficial em que é teste"] = piloto["paciente"].map(fold_teste).fillna("não está na divisão oficial")
    tab = pd.crosstab(piloto["grupo no piloto"], piloto["fold oficial em que é teste"])
    display(tab)
    salvar(piloto, "4_piloto_48_nos_folds.csv")
    dentro = piloto["paciente"].isin(fold_teste.index).sum()
    registrar(SEC, "INFO", f"{dentro} dos 48 pacientes do piloto estão na divisão oficial de "
                           f"{PAPEIS['paciente'].nunique()}, espalhados pelos folds: o piloto é outro experimento, "
                           "e os números dele não se misturam com os oficiais.")

Piloto: 48 pacientes; números de 1 a 50 que não estão nele: [28, 32]
[INFO] O piloto tem 48 pacientes (0001 a 0050, sem [28, 32]); o cruzamento com os folds oficiais fica PENDENTE.


## 5. As imagens da explicabilidade são mesmo do teste?

**O que é verificado.** Os mapas de explicação (Grad-CAM, IG, Adebayo) só valem se foram feitos em casos que o modelo **não viu no treino**. Para cada lista de casos:

- o paciente está na divisão oficial?
- se a lista diz qual fold ou modelo gerou o mapa, esse fold é o mesmo em que o paciente é **teste**?
- quantos casos há de cada grupo (acerto maligno, acerto benigno, falso positivo, falso negativo)?

Lembrete do rascunho do André: os mapas descrevem o **ensemble de 64 mm**, e não o sistema oficial completo.

In [9]:
SEC = "5. Explicabilidade"
fold_teste = (PAPEIS[PAPEIS["papel"] == "teste"].drop_duplicates("paciente").set_index("paciente")["fold"]
              if PAPEIS is not None else None)
algum = False
for nome, caminho in ARQS_XAI.items():
    arq = achar_arquivo(caminho, [nome.split("_")[0]])
    if arq is None:
        registrar(SEC, "PENDENTE", f"Lista '{nome}' não encontrada ({Path(caminho).name}).")
        continue
    algum = True
    casos = ler_csv(arq)
    c_pac, c_fold, c_grupo, c_caso = (achar_coluna(casos, p) for p in ["paciente", "fold", "grupo", "caso"])
    if c_pac is None and c_caso is not None and LINHAS is not None and "caso" in LINHAS:
        caso_para_pac = dict(zip(LINHAS["caso"].astype(str), LINHAS["paciente"]))
        casos["_pac_do_caso"] = casos[c_caso].astype(str).map(caso_para_pac)
        c_pac = "_pac_do_caso"
    if c_pac is None:
        registrar(SEC, "PENDENTE", f"'{nome}': não achei paciente nem um ID de caso ligável às previsões.")
        continue
    casos["_pac"] = casos[c_pac].map(normalizar_paciente)
    info = {"lista": nome, "casos": len(casos), "pacientes": casos["_pac"].nunique()}
    if fold_teste is not None:
        info["fora da divisão"] = int((~casos["_pac"].isin(fold_teste.index)).sum())
        if c_fold is not None:
            info["fold do mapa != fold em que é teste"] = int(
                (casos[c_fold].astype(str) != casos["_pac"].map(fold_teste)).sum())
    if c_grupo is not None:
        info["por grupo"] = casos[c_grupo].value_counts().to_dict()
    display(pd.DataFrame([info]))
    casos["fold em que é teste"] = casos["_pac"].map(fold_teste) if fold_teste is not None else np.nan
    salvar(casos.drop(columns="_pac"), f"5_{nome}_conferido.csv")
    problemas = info.get("fora da divisão", 0) + info.get("fold do mapa != fold em que é teste", 0)
    if fold_teste is None:
        registrar(SEC, "PENDENTE", f"'{nome}': {len(casos)} casos lidos, mas sem a divisão não dá para conferir.")
    elif c_fold is None:
        registrar(SEC, "INFO", f"'{nome}': {len(casos)} casos, {info['fora da divisão']} fora da divisão. "
                               "A lista não diz qual fold gerou o mapa; confirmar com o André que cada mapa "
                               "veio do modelo em que o paciente era teste.")
    else:
        registrar(SEC, "OK" if problemas == 0 else "ALERTA",
                  f"'{nome}': {len(casos)} casos; {problemas} com problema (fora da divisão ou mapa de um fold "
                  "em que o paciente era treino).")
if not algum:
    print("Nenhuma lista de explicabilidade encontrada.")

[PENDENTE] Lista 'xai_160' não encontrada (casos_xai.csv).
[PENDENTE] Lista 'adebayo_24' não encontrada (casos_adebayo.csv).
[PENDENTE] Lista 'exemplos_16' não encontrada (exemplos.csv).
Nenhuma lista de explicabilidade encontrada.


## 6. O que conta como maligno e o que é "um caso" (D6) + tamanho dos nódulos

**A regra oficial, para escrever igual em todo lugar:**

| Item | Regra |
|---|---|
| Classe positiva (maligno) | média das notas dos radiologistas **≥ 3,5** (não é biópsia) |
| Uma linha do arquivo | um **recorte** de um nódulo em uma fatia |
| Margem de erro (IC) | calculada **por paciente** |
| Visão principal | **sem** os nódulos ambíguos (média entre 2,5 e 3,5) |

**O efeito da troca de regra.** No notebook 03, maligno era "nota ≥ 3". A tabela abaixo mostra quanto a porcentagem de malignos muda entre as duas regras. É por isso que os números antigos e os novos não podem ser comparados diretamente.

**Tamanho dos nódulos (para a Débora).** Vem da coluna de diâmetro das previsões, se existir. Se não existir, é calculado pelas anotações originais (a maior área desenhada, convertida no diâmetro de um círculo de mesma área).

In [10]:
SEC = "6. D6 e tamanho"
if LINHAS is None or "media" not in LINHAS:
    registrar(SEC, "PENDENTE", "Sem a média de malignidade nas previsões, não dá para mostrar o efeito da regra.")
else:
    efeito = []
    for regra, limiar in [("antiga: nota >= 3", 3.0), ("oficial: média >= 3,5", LIMIAR_MALIGNO)]:
        mal = LINHAS["media"] >= limiar
        l = {"regra": regra, "% maligno (recortes)": round(100 * mal.mean(), 1)}
        if "chave_nodulo" in LINHAS:
            l["% maligno (nódulos)"] = round(100 * mal.groupby(LINHAS["chave_nodulo"]).max().mean(), 1)
        efeito.append(l)
    efeito = pd.DataFrame(efeito)
    display(efeito)
    salvar(efeito, "6_efeito_da_regra_de_maligno.csv")
    registrar(SEC, "INFO", "Regra oficial: maligno = média >= 3,5; linha = recorte; IC por paciente; "
                           "visão principal sem ambíguos (2,5 a 3,5). "
                           f"Malignos por recorte: {efeito.iloc[0, 1]}% na regra antiga, {efeito.iloc[1, 1]}% na oficial.")

# Tamanho dos nódulos
if LINHAS is not None and "diametro" in LINHAS:
    cols = [c for c in ["paciente", "nodulo", "chave_nodulo", "fold", "diametro", "media", "rotulo"] if c in LINHAS]
    chave = "chave_nodulo" if "chave_nodulo" in LINHAS else None
    tam = (LINHAS[cols].groupby(chave, as_index=False).first() if chave else LINHAS[cols].drop_duplicates())
    salvar(tam, "6_tamanho_nodulos.csv")
    registrar(SEC, "OK", f"Tamanho de {len(tam)} nódulos exportado das previsões oficiais (para a Débora).")
elif ANOT is not None and len(ANOT):
    try:
        import pydicom
        espac = {}
        def espacamento(serie):
            # [Adaptação Gabriel] lê o PixelSpacing de um DICOM da série (o XML não fica junto do DICOM aqui)
            if serie not in espac:
                dcm = SERIE_DCM.get(serie)
                espac[serie] = float(pydicom.dcmread(dcm, stop_before_pixels=True).PixelSpacing[0]) if dcm else np.nan
            return espac[serie]
        unidade = "mm"
    except ImportError:
        espacamento, unidade = (lambda xml: 1.0), "px"
    tam = ANOT[ANOT["malignancy"].notna()].copy()
    tam["diametro_" + unidade] = 2 * np.sqrt(tam["maior_area_px"] / np.pi) * tam["serie"].map(espacamento)
    tam = tam[["paciente", "serie", "leitor", "nodulo_xml", "malignancy", "n_fatias", "diametro_" + unidade]]
    salvar(tam, "6_tamanho_nodulos_por_anotacao.csv")
    registrar(SEC, "INFO", f"Tamanho calculado das anotações originais ({len(tam)} anotações, em {unidade}), "
                           "uma linha por radiologista. É um diâmetro por fatia, não o diâmetro 3D oficial. "
                           "Melhor: o André exportar a coluna de diâmetro do pipeline dele.")
else:
    registrar(SEC, "PENDENTE", "Sem coluna de diâmetro e sem a pasta do LIDC: tamanho dos nódulos pendente.")

[PENDENTE] Sem a média de malignidade nas previsões, não dá para mostrar o efeito da regra.


   -> salvo: evidencias/6_tamanho_nodulos_por_anotacao.csv (7034 linhas)
[INFO] Tamanho calculado das anotações originais (7034 anotações, em mm), uma linha por radiologista. É um diâmetro por fatia, não o diâmetro 3D oficial. Melhor: o André exportar a coluna de diâmetro do pipeline dele.


## 7. A ficha do radiologista "entrega a resposta"? (D5)

**O problema.** O MLP semântico e a fusão recebem como entrada a **ficha** que o radiologista preencheu: borda com espinhos, formato, textura etc. Só que quem preencheu a ficha é **a mesma pessoa** que deu a nota de malignidade, que é o rótulo. É como entregar o começo do laudo e pedir a conclusão.

**O teste.** Treinamos um modelo bem simples (regressão logística), **sem nenhuma imagem**, só com as 8 características da ficha, para prever a nota de malignidade **do mesmo radiologista**. A avaliação separa por paciente, usando os folds oficiais quando existem.

- Se esse modelo acerta quase tudo (AUC perto de 1), a ficha sozinha já resolve o problema.
- Nesse caso, o MLP e a fusão **não podem ser comparados** com o modelo oficial, que só vê a imagem.

Precisa da pasta do LIDC original (os XML).

In [11]:
SEC = "7. Vazamento (D5)"
FICHA = ["subtlety", "internalStructure", "calcification", "sphericity", "margin", "lobulation", "spiculation", "texture"]

def logistica_oof(X, y, grupos, passos=3000, taxa=0.1, l2=1e-3):
    """Regressão logística simples (numpy). Para cada grupo, treina nos outros e prevê nele."""
    oof = np.full(len(y), np.nan)
    for g in np.unique(grupos):
        tr, te = grupos != g, grupos == g
        mu, dp = X[tr].mean(0), X[tr].std(0) + 1e-9
        A = np.c_[np.ones(tr.sum()), (X[tr] - mu) / dp]
        B = np.c_[np.ones(te.sum()), (X[te] - mu) / dp]
        w = np.zeros(A.shape[1])
        for _ in range(passos):
            p = 1 / (1 + np.exp(-A @ w))
            w -= taxa * (A.T @ (p - y[tr]) / len(p) + l2 * np.r_[0, w[1:]])
        oof[te] = 1 / (1 + np.exp(-B @ w))
    return oof

if ANOT is None or not len(ANOT):
    registrar(SEC, "PENDENTE", "Sem a pasta do LIDC original, o teste da ficha do radiologista fica pendente.")
else:
    d = ANOT.dropna(subset=FICHA + ["malignancy", "paciente"]).copy()
    # grupo = fold oficial do paciente; sem fold oficial, grupo fixo pelo número do paciente (5 grupos)
    reserva = d["paciente"].map(lambda p: "g" + str(int(p[-4:]) % 5))
    if PAPEIS is not None:
        ft = PAPEIS[PAPEIS["papel"] == "teste"].drop_duplicates("paciente").set_index("paciente")["fold"]
        oficial = d["paciente"].map(ft).astype(object)
    else:
        oficial = pd.Series(np.nan, index=d.index, dtype=object)
    d["grupo"] = oficial.where(oficial.notna(), reserva).astype(str)

    resultados = []
    for visao, filtro, alvo in [("principal (sem nota 3)", d["malignancy"] != 3, d["malignancy"] >= 4),
                                ("todos (nota 3 = benigno)", d["malignancy"] > 0, d["malignancy"] >= 4)]:
        sub = d[filtro]
        y = alvo[filtro].astype(int).to_numpy()
        X = sub[FICHA].to_numpy(dtype=float)
        if len(np.unique(y)) < 2 or sub["paciente"].nunique() < 5:
            resultados.append({"visão": visao, "anotações": len(sub), "AUC da ficha (sem imagem)": np.nan})
            continue
        oof = logistica_oof(X, y, sub["grupo"].to_numpy())
        l = {"visão": visao, "anotações": len(sub), "pacientes": sub["paciente"].nunique(),
             "% maligno": round(100 * y.mean(), 1), "AUC da ficha (sem imagem)": round(auc(y, oof), 3)}
        for c in FICHA:   # cada característica sozinha
            a = auc(y, sub[c].to_numpy(dtype=float))
            l[f"AUC só {c}"] = round(max(a, 1 - a), 3)
        resultados.append(l)
    res = pd.DataFrame(resultados)
    display(res.T)
    salvar(res, "7_vazamento_ficha_radiologista.csv")
    a_principal = res.loc[0, "AUC da ficha (sem imagem)"]
    if pd.isna(a_principal):
        registrar(SEC, "PENDENTE", "Poucos dados para o teste da ficha.")
    else:
        alerta = a_principal >= 0.85
        registrar(SEC, "ALERTA" if alerta else "INFO",
                  f"Só com a ficha do radiologista, sem ver a imagem, um modelo simples chega a AUC {a_principal:.3f} "
                  f"({res.loc[0, 'pacientes']} pacientes). "
                  + ("A ficha praticamente entrega a resposta: MLP semântico e fusão devem ficar como "
                     "'não comparáveis' com o modelo oficial (só imagem)." if alerta else
                     "A ficha não resolve o problema sozinha."))
    if PAPEIS is None:
        registrar(SEC, "INFO", "Teste feito sem a divisão oficial (grupos fixos pelo número do paciente): resultado provisório.")

,0,1
visão,principal (sem nota 3),todos (nota 3 = benigno)
anotações,2773,4552
pacientes,531,559
% maligno,41.4,25.2
AUC da ficha (sem imagem),0.908,0.827
AUC só subtlety,0.731,0.725
AUC só internalStructure,0.501,0.5
AUC só calcification,0.649,0.571
AUC só sphericity,0.592,0.553
AUC só margin,0.668,0.582


   -> salvo: evidencias/7_vazamento_ficha_radiologista.csv (2 linhas)
[ALERTA] Só com a ficha do radiologista, sem ver a imagem, um modelo simples chega a AUC 0.908 (531 pacientes). A ficha praticamente entrega a resposta: MLP semântico e fusão devem ficar como 'não comparáveis' com o modelo oficial (só imagem).
[INFO] Teste feito sem a divisão oficial (grupos fixos pelo número do paciente): resultado provisório.


**Como ler.** "AUC" vai de 0,5 (chute) a 1 (acerta sempre). Para comparar, o modelo oficial, que vê a imagem, tem AUC 0,878. Se a ficha sozinha passa disso **sem olhar a imagem**, ela carrega a resposta. A coluna "AUC só ..." mostra quais características da ficha mais "entregam" a nota.

## Resumo final

Junta tudo em `evidencias/resumo_auditoria.md` e registra o ambiente (versões, data, impressões digitais dos arquivos) em `evidencias/ambiente.txt`.

- **OK:** a checagem passou.
- **ALERTA:** algo precisa ser explicado.
- **PENDENTE:** faltou arquivo ou coluna.
- **INFO:** é só descrição.

In [12]:
resumo = pd.DataFrame(RESUMO, columns=["seção", "status", "o que significa"])
display(resumo)

texto = ["# Resumo da auditoria do dataset oficial (Sprint 5, P3)", "",
         f"Rodado em {datetime.datetime.now():%Y-%m-%d %H:%M}.", "",
         "| Seção | Status | O que significa |", "|---|---|---|"]
texto += [f"| {s} | {st} | {f} |" for s, st, f in RESUMO]
(PASTA_SAIDA / "resumo_auditoria.md").write_text("\n".join(texto) + "\n", encoding="utf-8")

amb = [f"data: {datetime.datetime.now().isoformat(timespec='seconds')}",
       f"python: {platform.python_version()} | pandas: {pd.__version__} | numpy: {np.__version__}",
       f"sistema: {platform.platform()}",
       f"LIMIAR_MALIGNO={LIMIAR_MALIGNO} AMBIGUO={AMBIGUO} SHA256_ESPERADO={SHA256_ESPERADO}",
       f"PASTA_LIDC: {'configurada' if PASTA_LIDC else 'não configurada'}"]
for rotulo, arq in [("divisão", arq_div), ("previsões", arq_prev)]:
    amb.append(f"{rotulo}: {arq.name} sha256={sha256_arquivo(arq)}" if arq else f"{rotulo}: não encontrado")
(PASTA_SAIDA / "ambiente.txt").write_text("\n".join(amb) + "\n", encoding="utf-8")

print("\nArquivos gerados em", PASTA_SAIDA)
for p in sorted(PASTA_SAIDA.iterdir()):
    print("  ", p.name)

,seção,status,o que significa
0,1. Separação,PENDENTE,"Sem o arquivo da divisão, não dá para conferir..."
1,2. Composição,PENDENTE,"Sem o arquivo de previsões (com paciente), não..."
2,3. Descartes,PENDENTE,Funil sem a divisão oficial: só mostra o que e...
3,3. Descartes,PENDENTE,"Sem o arquivo de previsões, a correspondência ..."
4,4. Origem dos 48,INFO,"O piloto tem 48 pacientes (0001 a 0050, sem [2..."
5,5. Explicabilidade,PENDENTE,Lista 'xai_160' não encontrada (casos_xai.csv).
6,5. Explicabilidade,PENDENTE,Lista 'adebayo_24' não encontrada (casos_adeba...
7,5. Explicabilidade,PENDENTE,Lista 'exemplos_16' não encontrada (exemplos.c...
8,6. D6 e tamanho,PENDENTE,"Sem a média de malignidade nas previsões, não ..."
9,6. D6 e tamanho,INFO,Tamanho calculado das anotações originais (703...



Arquivos gerados em C:\Users\Administrator\Documents\Biopark\Projeto-Integrador-3\PI3-Grupo-03\auditoria_dataset\evidencias
   3_funil_pacientes.csv
   6_tamanho_nodulos_por_anotacao.csv
   7_vazamento_ficha_radiologista.csv
   ambiente.txt
   resumo_auditoria.md
